In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [12]:
from src.loader import load_league_season
nba_25_26 = load_league_season("nba", "25-26")



Loaded: advanced.csv
Shape: (734, 30)
Loaded: playbyplay.csv
Shape: (734, 26)
Loaded: perposs.csv
Shape: (734, 34)
Loaded: shooting.csv
Shape: (584, 32)


In [13]:
from src.cleaner import clean_stat_table

advanced = clean_stat_table(nba_25_26["advanced"])
perposs = clean_stat_table(nba_25_26["perposs"])
shooting = clean_stat_table(nba_25_26["shooting"])
playbyplay = clean_stat_table(nba_25_26["playbyplay"])

In [14]:
from src.cleaner import clean_stat_table

advanced = clean_stat_table(nba_25_26["advanced"])
perposs = clean_stat_table(nba_25_26["perposs"])
shooting = clean_stat_table(nba_25_26["shooting"])
playbyplay = clean_stat_table(nba_25_26["playbyplay"])

In [15]:
from src.profiles import build_player_profiles

profiles = build_player_profiles(
    advanced=advanced,
    perposs=perposs,
    shooting=shooting,
    playbyplay=playbyplay
)

profiles.shape

(734, 98)

In [16]:
from src.profile_cleanup import clean_profiles

profiles_clean = clean_profiles(profiles)

profiles_clean.shape

(734, 94)

In [17]:
from models.model import build_model_dataset
from src.database import save_df_to_sqlite
from src.profile_cleanup import clean_profiles

model_df = build_model_dataset(
    profiles_clean,
    min_minutes=500
)

model_df.shape
model_df.columns.tolist()

['player',
 'tm',
 'pos',
 'age',
 'g',
 'gs',
 'mp',
 'player_id',
 'pts',
 'ts_pct',
 'efg_pct',
 'fg_pct',
 '2p_pct',
 '3p_pct',
 'ft_pct',
 'three_point_attempt_rate',
 'free_throw_rate',
 'dist',
 '0_3',
 '3_10',
 '10_16',
 '16_3p',
 '0_3_2',
 '3_10_2',
 '10_16_2',
 '16_3p_2',
 'ast',
 'assist_pct',
 'tov',
 'turnover_pct',
 'obpm',
 'ows',
 'ortg',
 'orb',
 'drb',
 'trb',
 'off_reb_pct',
 'def_reb_pct',
 'total_reb_pct',
 'stl',
 'blk',
 'steal_pct',
 'block_pct',
 'pf',
 'drtg',
 'dbpm',
 'dws',
 'per',
 'ws',
 'ws_per_48',
 'bpm',
 'vorp',
 'on_off',
 'pg_pct',
 'sg_pct',
 'sf_pct',
 'pf_pct',
 'c_pct']

In [18]:
profiles_clean.to_csv("../DATA/processed/nba_profiles_master.csv", index=False)
model_df.to_csv("../DATA/processed/nba_25_26_model_dataset_v2.csv", index=False)

save_df_to_sqlite(
    model_df,
    "../DATA/database/parallel_hoops_v2.db",
    "nba_model_dataset_25_26_v2"
)

Saved 435 rows to ../DATA/database/parallel_hoops_v2.db table: nba_model_dataset_25_26_v2


In [19]:
from importlib import reload
import src.similarity_engine
reload(src.similarity_engine)

from src.similarity_engine import find_similar_players, similarity_feature_report

similarity_feature_report(model_df)

{'scoring': {'weight': 0.175,
  'available': ['pts',
   'ts_pct',
   'efg_pct',
   'fg_pct',
   '2p_pct',
   '3p_pct',
   'ft_pct',
   'three_point_attempt_rate',
   'free_throw_rate',
   'dist',
   '0_3',
   '3_10',
   '10_16',
   '16_3p'],
  'missing': []},
 'playmaking': {'weight': 0.175,
  'available': ['ast',
   'assist_pct',
   'tov',
   'turnover_pct',
   'obpm',
   'ows',
   'ortg'],
  'missing': ['usage_pct']},
 'individual_defense': {'weight': 0.175,
  'available': ['stl', 'blk', 'steal_pct', 'block_pct', 'pf'],
  'missing': []},
 'team_defense': {'weight': 0.175,
  'available': ['drtg', 'dbpm', 'dws', 'drb', 'def_reb_pct'],
  'missing': []},
 'impact': {'weight': 0.2,
  'available': ['per', 'ws', 'ws_per_48', 'bpm', 'vorp', 'on_off'],
  'missing': []},
 'position': {'weight': 0.1,
  'available': ['pg_pct', 'sg_pct', 'sf_pct', 'pf_pct', 'c_pct'],
  'missing': []}}

In [42]:
from src.normalize import normalize_by_league

model_df["league"] = "NBA"
model_df["season"] = "25-26"

nba_normalized = normalize_by_league(model_df)

nba_normalized.to_csv(
    "../DATA/processed/nba_25_26_normalized.csv" , 
    index=False
)


In [ ]:
from scipy.spatial.distance import euclidean

cols = [
    "drtg",
    "dbpm",
    "dws",
    "def_reb_pct"
]

lamelo = nba_normalized[
    nba_normalized["player"] == "LaMelo Ball"
][cols].iloc[0]

cade = nba_normalized[
    nba_normalized["player"] == "Cade Cunningham"
][cols].iloc[0]

distance = euclidean(lamelo, cade)

distance

In [ ]:
from importlib import reload
import src.similarity_engine
reload(src.similarity_engine)

from src.similarity_engine import find_similar_players

find_similar_players(
    "Jalen Johnson",
    nba_normalized,
    top_n=10
)